(year-specific-workflows-copy)=
# Orchestrate year-specific workflows across survey years

This notebook presents an **example implementation** of multi-year orchestration for FEAT year scripts, grounded directly in the `year_specific/` source files. The discussion keeps the same technical framing used across this documentation section: workflow-stage composition follows the assumptions in the [workflow overview](../workflows.md), orchestration behavior follows the execution model in the [orchestration guide](../orchestrating_multiple_workflows.md), and CLI argument flow follows the hook patterns established in the [CLI walkthrough](cli_from_scratch.ipynb). Where runtime routing intersects with externalized parameter design, references are made to the [JSON/YAML configuration tutorial](workflow_configuration_json_yaml_only.ipynb), while the examples here remain centered on the current dispatcher-plus-year-script architecture.

## Architecture overview

The orchestration model uses a single dispatcher to route execution to a selected year script (`hake_1995.py` through `hake_2023.py`), while shared CLI hooks expose runtime controls such as verbosity and report-comparison behavior.

![Year-specific orchestration flow](../../_static/year_specific_orchestration_flow.svg)

In practice, the flow starts from one of three surfaces: direct CLI invocation, wrapper-script execution, or notebook orchestration. After the entrypoint call reaches `year_specific_workflow.py`, the dispatcher resolves the year target, forwards passthrough arguments, and runs the selected script through a consistent ingestion-to-reporting sequence. Shared utilities in `cli_utils.py` provide reusable hook behavior inside each year script, which keeps orchestration logic centralized without flattening year-specific parameter definitions.

![General year-script processing steps](../../_static/year_specific_processing_steps.svg)

Across `hake_1995.py` through `hake_2023.py`, the processing spine remains stable: parameter entry and runtime flags lead into ingestion, preprocessing, inversion, geostatistics, and report generation, with optional comparison outputs when compare mode is enabled.

## Source files and responsibilities

The orchestration contract is implemented through three cooperating source layers. The dispatcher layer in `year_specific_workflow.py` parses `--year`, `--verbose`, and `--compare`, routes to `hake_<year>.py` using `runpy.run_path(...)`, and preserves passthrough arguments through `parse_known_args()`. The shared-hook layer in `cli_utils.py` defines `get_verbose()` and `get_compare()`, which year scripts consume to adapt runtime behavior based on CLI state. The year-script layer (`hake_1995.py` through `hake_2023.py`) contains year-specific file mappings and parameters while still following the same FEAT stage pattern, which allows local specialization without fragmenting the orchestration interface.

## Workflow parameters across year scripts

Parameter blocks in `year_specific/hake_*.py` follow a shared schema even when year-specific files diverge on source files or survey composition. The inventory below reflects variables defined before the `START OF PROCESSING SCRIPT` marker across all available year scripts (`1995`, `1998`, `2001`, `2003`, `2005`, `2007`, `2009`, `2011`, `2012`, `2013`, `2015`, `2017`, `2019`, `2021`, `2023`).

| Variable(s) | Functional role | Year coverage |
|---|---|---|
| `VERBOSE` | Enables stage-level logger output (`logging.INFO`) when the verbose hook is active. | All listed years |
| `COMPARE`, `ECHOPRO_REPORTS_DIR`, `COMPARISONS_DIR`, `SHOW_PLOT` | Enables and configures EchoPro-vs-Echopop comparison branch outputs. | All listed years |
| `DATA_ROOT`, `REPORTS_DIR` | Root pathing for input assets and report/export destinations. | All listed years |
| `NASC_EXPORTS_FILES`, `NASC_EXPORTS_SHEET` | NASC input workbook path and worksheet target used by ingestion. | All listed years |
| `NASC_PREPROCESSED` | Switches between preformatted NASC input and raw-export processing pathways. | `2003`, `2005`, `2007`, `2009`, `2011`, `2012`, `2013`, `2015`, `2017`, `2019`, `2021`, `2023` |
| `REMOVE_AGE1`, `AGE1_DOMINATED_HAULS` | Controls age-1 exclusion and haul-level override behavior for age composition handling. | All listed years |
| `BIODATA_FILE`, `BIODATA_SHEETS`, `BIODATA_SHIP_SPECIES` | Defines biological input file(s), sheet map, and ship/survey/species parsing contract. | All listed years |
| `SPECIES_ID`, `SHIP_US`, `SURVEY_US` | Core U.S.-side species and survey identifiers used by UID construction and filtering. | `SPECIES_ID`: all years; `SHIP_US`/`SURVEY_US`: all years except `2003` |
| `SHIP_CAN`, `SURVEY_CAN`, `CAN_HAUL_OFFSET` | Canadian survey IDs and haul-offset handling for joint-survey harmonization. | `SHIP_CAN`/`SURVEY_CAN`: `1998`, `2001`, `2003`, `2009`, `2011`, `2012`, `2013`, `2015`, `2017`, `2019`, `2021`, `2023`; `CAN_HAUL_OFFSET`: same set excluding `2003` |
| `SURVEY_FILTER` | Applies survey-ID filtering logic in mixed-source ingestion windows. | `1995`, `1998`, `2001`, `2003`, `2005`, `2007`, `2009`, `2011`, `2012`, `2013` |
| `HAUL_STRATA_FILE`, `HAUL_STRATA_SHEETS` | Haul-based stratification source and sheet mapping (`ks`, `inpfc`). | All listed years |
| `GEOSTRATA_FILE`, `GEOSTRATA_SHEETS` | Geographic stratification source and sheet mapping (`ks`, `inpfc`). | All listed years |
| `KRIGING_MESH_FILE`, `KRIGING_MESH_SHEET` | Spatial mesh source used by kriging/grid conversion stages. | All listed years |
| `KRIGING_VARIOGRAM_PARAMETERS_FILE`, `KRIGING_VARIGORAM_PARAMETERS_SHEET` | Variogram/kriging settings source and worksheet key. | All listed years |
| `OPTIMIZE_VARIOGRAM` | Toggles variogram optimization versus fixed-parameter operation. | All listed years |
| `ISOBATH_FILE`, `ISOBATH_SHEET` | Isobath coordinate-transform source used in geostatistical stages. | All listed years |
| `TRANSECT_BOUNDARY_FILE`, `TRANSECT_BOUNDARY_SHEET` | Transect-boundary lookup for transect/regional assignment steps. | `1995`, `1998`, `2001`, `2003`, `2005`, `2007`, `2009`, `2011`, `2012`, `2013` |
| `TRANSECT_REGION_HAUL_FILE`, `TRANSECT_REGION_HAUL_SHEET` | Region/haul mapping inputs used in mid-period transect regionalization. | `TRANSECT_REGION_HAUL_FILE`: `2003`, `2005`, `2007`, `2009`, `2011`, `2012`, `2013`; `TRANSECT_REGION_HAUL_SHEET`: same set excluding `2011` |

This cross-year pattern keeps dispatcher behavior stable while allowing each year script to refine ingestion and survey harmonization details without changing the external orchestration interface.

## Dispatcher CLI design

The dispatcher is intentionally thin: it resolves a year target, forwards runtime arguments, and delegates execution without embedding survey-domain logic. This separation keeps routing behavior stable while allowing year scripts to evolve independently.

```python
import argparse
import runpy
import sys
import os

parser = argparse.ArgumentParser()
parser.add_argument("--year", required=True, help="Which workflow script to run (e.g. hake_1995)")
parser.add_argument("--verbose", action="store_true", help="Enable verbose logging")
parser.add_argument("--compare", action="store_false", help="Compare output reports to EchoPro")
args, unknown = parser.parse_known_args()

sys.argv = [args.year + ".py"] + unknown
if args.verbose:
    sys.argv.append("--verbose")
if args.compare:
    sys.argv.append("--compare")

script_path = os.path.join(os.path.dirname(__file__), f"{args.year}.py")
runpy.run_path(script_path, run_name="__main__")
```

Two implementation details matter for orchestration reliability: `parse_known_args()` preserves flags intended for downstream scripts, and `runpy.run_path(...)` executes the selected year module in-process without requiring duplicated launcher code per survey year.

## Shared hooks and year-script integration

Year scripts use shared utility hooks to read CLI state and control runtime behavior, especially logging and comparison pathways. Centralizing these checks in one utility module reduces drift across year files and keeps hook semantics consistent.

```python
# cli_utils.py
import argparse

def get_verbose():
    parser = argparse.ArgumentParser()
    parser.add_argument("--verbose", action="store_true", help="Enable verbose logging")
    args, _ = parser.parse_known_args()
    return args.verbose

def get_compare():
    parser = argparse.ArgumentParser()
    parser.add_argument("--compare", action="store_false", help="Compare generated reports to EchoPro")
    args, _ = parser.parse_known_args()
    return args.compare
```

```python
# pattern used in year scripts
from echopop.workflows.nwfsc_feat import cli_utils

try:
    VERBOSE = cli_utils.get_verbose()
except Exception:
    VERBOSE = True

try:
    COMPARE = cli_utils.get_compare()
except Exception:
    COMPARE = False
```

The try/except fallback pattern allows the same year script to run cleanly from CLI, wrapper, and interactive contexts, while still defaulting to predictable behavior when flags are absent.

## CLI usage examples by platform

In each platform variant below, `--year` selects the script target and optional flags tune runtime behavior without changing year-file source code. The commands are intentionally parallel so users can move between shells without learning a different orchestration contract.

::::{tab-set}
:::{tab-item} Windows CMD
```bat
python year_specific_workflow.py --year hake_2023
python year_specific_workflow.py --year hake_2021 --verbose
python year_specific_workflow.py --year hake_2019 --verbose --compare
```
:::
:::{tab-item} Windows PowerShell
```powershell
python year_specific_workflow.py --year hake_2023
python year_specific_workflow.py --year hake_2021 --verbose
python year_specific_workflow.py --year hake_2019 --verbose --compare
```
:::
:::{tab-item} Linux/macOS bash
```bash
python3 year_specific_workflow.py --year hake_2023
python3 year_specific_workflow.py --year hake_2021 --verbose
python3 year_specific_workflow.py --year hake_2019 --verbose --compare
```
:::
::::

## Notebook-driven orchestration

A notebook can serve as a lightweight orchestration surface when live logs, incremental diagnostics, and reproducible command history are needed in one place. The pattern below launches the same dispatcher command used in terminal workflows and streams output in real time, so notebook execution preserves operational visibility rather than hiding stage progress until process completion.

```python
import subprocess

year = 2023
cmd = ["python", "year_specific_workflow.py", "--year", f"hake_{year}", "--verbose"]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="")
```

In [ ]:
import subprocess

year = 1995

process = subprocess.Popen(
    ["python", "year_specific_workflow.py", "--year", f"hake_{year}", "--verbose"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

## Batch orchestration across multiple years

For repeated execution across survey years, wrapper scripts provide a practical control layer that reuses the same dispatcher contract instead of duplicating orchestration logic in every environment. The examples below keep failure handling explicit so a run can be audited quickly after completion.

::::{tab-set}
:::{tab-item} Windows .bat
```bat
@echo off
setlocal enabledelayedexpansion

set YEARS=1995 1998 2001 2003

for %%Y in (%YEARS%) do (
    echo [INFO] Running hake_%%Y ...
    python year_specific_workflow.py --year hake_%%Y --verbose
    if errorlevel 1 (
        echo [ERROR] Failed year %%Y
    )
)
```
:::
:::{tab-item} Windows PowerShell (.ps1)
```powershell
$years = @(1995, 1998, 2001, 2003)
$failedYears = @()

foreach ($year in $years) {
    Write-Host "[INFO] Running hake_$year ..."
    python year_specific_workflow.py --year "hake_$year" --verbose

    if ($LASTEXITCODE -ne 0) {
        Write-Warning "Failed year $year"
        $failedYears += $year
    }
}

if ($failedYears.Count -gt 0) {
    Write-Host "Failed years: $($failedYears -join ', ')"
}
```
:::
:::{tab-item} Linux/macOS bash (.sh)
```bash
#!/usr/bin/env bash
set -u

years=(1995 1998 2001 2003)
failed_years=()

for year in "${years[@]}"; do
  echo "[INFO] Running hake_${year} ..."
  if ! python3 year_specific_workflow.py --year "hake_${year}" --verbose; then
    echo "[ERROR] Failed year ${year}"
    failed_years+=("$year")
  fi
done

if [ ${#failed_years[@]} -gt 0 ]; then
  echo "Failed years: ${failed_years[*]}"
fi
```
:::
::::

## Practical notes on CLI flags and logging

In this implementation, `--year` is the routing key that determines which year script is executed, while `--verbose` controls whether detailed stage progress is emitted to the console by scripts configured to honor that flag. The compare pathway is handled through the shared compare hook used by both dispatcher and year scripts, so behavior remains consistent across CLI, notebook, and wrapper-based runs. Because year scripts emit stage-level progress messages (for example with `logging.info(...)`), orchestration layers should preserve stdout/stderr streaming to support live monitoring, rapid failure diagnosis, and run-trace validation.

## Read next

For deeper grounding in stage composition, the [workflow overview](../workflows.md) details how analysis phases are defined and exchanged. For broader orchestration patterns and execution trade-offs, the [orchestration guide](../orchestrating_multiple_workflows.md) provides the surrounding framework for dispatcher-based designs. For parser behavior and hook evolution, the [CLI walkthrough](cli_from_scratch.ipynb) extends the interface patterns used here. For externalized runtime contracts, the [JSON/YAML configuration tutorial](workflow_configuration_json_yaml_only.ipynb) shows how parameter routing can move out of scripts while preserving reproducibility. For post-run synthesis and reporting across years, the [cross-year comparison notebook](cross_year_comparisons.ipynb) continues the same workflow lineage.